# ll-hls4ml tensor-v2 long training on Kaggle

## Experiment goal

This notebook uses one Kaggle accelerator session for a **matched long-run
comparison**, rather than a broad ablation sweep:

1. a pooled MLP control using the enriched tensor-v2 representation;
2. a three-layer heterogeneous GATv2 with the same readout, heads, loss, split,
   and target treatment.

The control is essential: a long GAT run by itself cannot show whether graph
message passing adds value beyond typed node features and graph pooling.

Default wall-clock allocation is 2 hours for the MLP and 8.5 hours for GATv2,
leaving margin within a 12-hour session for setup, validation, final inference,
and artifact packaging. Both jobs checkpoint every epoch. If `timeout` stops a
job, the notebook evaluates its best validation-SMAPE checkpoint afterward.

> **Tensor-v2 requirement:** the current public Hub repository name below was
> used by the older notebook and may still contain tensor-v1. Set
> `TENSOR_REPO_ID`/`TENSOR_REVISION` to the uploaded v2 snapshot. The validation
> cell deliberately refuses tensors without the v2 pragma, block, and synthesis
> context schema.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import torch

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "ll-hls4ml"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/ll-hls4ml.git"
# Pin this after pushing the notebook-supporting code for exact reproducibility.
REPO_REF = "b181f7fc84dd3fb6946d4e924de1f0207254e028"

# IMPORTANT: point this at the uploaded tensor-v2 snapshot.
TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-pragmas-v2"
TENSOR_REVISION = "8abfe2314ef9144eb8fd612d30babfcfca173c51"
TENSOR_DIR = WORK / "tensors_v2"

# To resume in a later Kaggle session, attach the previous results as an input
# and set this to the directory containing the checkpoint files.
# Example: Path("/kaggle/input/models/<owner>/<model>/<framework>/<variant>/<version>")
PREVIOUS_RESULTS_ROOT = None

SEED = 42
RUN_MLP_CONTROL = False
MLP_TRAIN_BUDGET = "120m"
GAT_TRAIN_BUDGET = "660m"

GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator before running."
print("PyTorch:", torch.__version__)
print("CUDA devices:", GPU_COUNT)
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index))


## Install dependencies and checkout the training code

In [ ]:
# Kaggle already supplies CUDA-enabled PyTorch.
%pip install -q torch-geometric huggingface_hub pyyaml pandas matplotlib


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"],
        check=True,
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", REPO_REF],
    check=True,
)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()
print("ll-hls4ml commit:", commit)

sys.path.insert(0, str(REPO_DIR / "src"))


## Download and validate tensor-v2

In [ ]:
from huggingface_hub import login, snapshot_download

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)

print("Downloading", TENSOR_REPO_ID, "revision", TENSOR_REVISION or "main")
snapshot_download(
    repo_id=TENSOR_REPO_ID,
    repo_type="dataset",
    revision=TENSOR_REVISION,
    token=hf_token,
    local_dir=str(TENSOR_DIR),
    allow_patterns=["*.pt", "*.json"],
)
print("Tensor snapshot downloaded to", TENSOR_DIR)


In [ ]:
import json
import torch

from ll_hls4ml.io.schema import (
    BLOCK_FEATURE_SIZE,
    GRAPH_CONTEXT_CATEGORICAL_VOCABS,
    GRAPH_CONTEXT_NUMERIC_KEYS,
    PRAGMA_FEATURE_SIZE,
)

tensor_files = sorted(TENSOR_DIR.rglob("*.pt"))
assert tensor_files, f"No .pt tensors found under {TENSOR_DIR}"

labels_path = TENSOR_DIR / "labels.json"
assert labels_path.is_file(), "tensor-v2 labels.json is missing"
labels_payload = json.loads(labels_path.read_text())
label_count = len(labels_payload.get("labels", {}))

sample = torch.load(tensor_files[0], map_location="cpu", weights_only=False)
errors = []
if sample["pragma"].x.shape[1] != PRAGMA_FEATURE_SIZE:
    errors.append(
        f"pragma width {sample['pragma'].x.shape[1]} != v2 width "
        f"{PRAGMA_FEATURE_SIZE}"
    )
if sample["block"].x.shape[1] != BLOCK_FEATURE_SIZE:
    errors.append(
        f"block width {sample['block'].x.shape[1]} != "
        f"{BLOCK_FEATURE_SIZE}"
    )
if not hasattr(sample, "graph_context_categorical"):
    errors.append("graph_context_categorical is absent")
elif sample.graph_context_categorical.shape[-1] != len(
    GRAPH_CONTEXT_CATEGORICAL_VOCABS
):
    errors.append("categorical synthesis-context width is incorrect")
if not hasattr(sample, "graph_context_numeric"):
    errors.append("graph_context_numeric is absent")
elif sample.graph_context_numeric.shape[-1] != len(
    GRAPH_CONTEXT_NUMERIC_KEYS
):
    errors.append("numeric synthesis-context width is incorrect")

assert not errors, (
    "Downloaded tensors are not tensor-v2:\n- " + "\n- ".join(errors)
)

vocab_candidates = [
    TENSOR_DIR / "vocab.json",
    REPO_DIR / "artifacts" / "vocab" / "vocab.json",
]
VOCAB_PATH = next(
    (path for path in vocab_candidates if path.is_file()),
    None,
)
assert VOCAB_PATH is not None, (
    "vocab.json is missing. Include the tensor-v2 build vocabulary in the "
    "Hub snapshot."
)

print("Tensor files:", len(tensor_files))
print("Indexed labels:", label_count)
print("Sample:", tensor_files[0].relative_to(TENSOR_DIR))
print("Pragma width:", sample["pragma"].x.shape[1])
print("Block width:", sample["block"].x.shape[1])
print("Context widths:", sample.graph_context_categorical.shape[-1],
      sample.graph_context_numeric.shape[-1])
print("Vocabulary:", VOCAB_PATH)


## Matched long-run configurations

Both models use:

- official dataset splits;
- empirical distributed sampling;
- multi-statistic, count-aware pooling and the global feature shortcut;
- no synthesis-context branch (context is retained in tensor-v2 but is confounded
  with dataset cohort in the current training set);
- separate resource/timing towers;
- DSP/BRAM hurdle outputs;
- robust regression in `log1p(target)` space.

`batch_size=1` is per GPU. With two Kaggle GPUs, DDP gives a global batch of two.
This conservative choice avoids GAT attention OOMs on the largest graphs.

Every completed epoch writes an optimizer backup. Re-running either training
cell automatically resumes the local backup. To resume in a later Kaggle
session, attach the prior packaged results and set `PREVIOUS_RESULTS_ROOT` in
the settings cell.


In [ ]:
import json
import shutil

KERNEL_TYPES = [
    "2layer",
    "3layer",
    "conv1d",
    "conv2d",
    "dense_latency",
    "dense_resource",
    "rule4ml",
]

common = {
    "tensor_dir": str(TENSOR_DIR),
    "vocab_path": str(VOCAB_PATH),
    "results_dir": str(RESULTS_DIR),
    "kernel_types": KERNEL_TYPES,
    "seed": SEED,
    "split_strategy": "official_or_stratified",
    # WeightedRandomSampler is not sharded by the current DDP loader.
    # Use the same empirical distributed sampler for both matched models.
    "family_balanced_sampling": False,
    "batch_size": 1,
    "num_workers": 0,
    "epochs": 400,
    "patience": 30,
    "weight_decay": 1e-4,
    "dropout": 0.15,
    "pool": "multi",
    "use_global_features": True,
    "use_context": False,
    "split_heads": True,
    "hurdle_heads": True,
    "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle",
    "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape",
    "verbose": 2,
}

mlp_config = {
    **common,
    "experiment_name": "kaggle_v2_mlp_long_seed42",
    "model": "mlp",
    "checkpoint_dir": str(
        RESULTS_DIR / "kaggle_v2_mlp_long_seed42" / "checkpoints"
    ),
    "learning_rate": 1e-3,
    "hidden_dim": 64,
    "num_layers": 2,
    "num_var_embed_layers": 2,
    "node_aggr": "concat",
}

gat_config = {
    **common,
    "experiment_name": "kaggle_v2_gatv2_1head_3layer_long_seed42",
    "model": "hetero_gat",
    "checkpoint_dir": str(
        RESULTS_DIR / "kaggle_v2_gatv2_3layer_long_seed42" / "checkpoints"
    ),
    "learning_rate": 5e-4,
    "hidden_dim": 64,
    "num_layers": 3,
    "aggr": "sum",
}

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _previous_search_root():
    if PREVIOUS_RESULTS_ROOT is None:
        return None
    root = Path(PREVIOUS_RESULTS_ROOT)
    archives = sorted(root.rglob("ll_hls4ml_long_run_results.zip"))
    if archives:
        if len(archives) > 1:
            raise RuntimeError(f"Multiple previous result archives: {archives}")
        extracted = WORK / "previous_results"
        if not extracted.is_dir():
            shutil.unpack_archive(archives[0], extracted)
            print("Extracted previous results:", archives[0])
        return extracted
    return root

def _find_previous(name):
    root = _previous_search_root()
    if root is None:
        return None
    matches = sorted(root.rglob(name))
    if len(matches) > 1:
        raise RuntimeError(f"Multiple resume candidates for {name}: {matches}")
    return matches[0] if matches else None

def write_config(config):
    path = CONFIG_DIR / f"{config['experiment_name']}.json"
    checkpoint_dir = Path(config["checkpoint_dir"])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    backup_name = f"{config['experiment_name']}_backup.pt"
    best_name = f"{config['experiment_name']}_checkpoint.pt"
    backup = checkpoint_dir / backup_name
    best = checkpoint_dir / best_name

    previous_backup = _find_previous(backup_name)
    previous_best = _find_previous(best_name)
    if not backup.is_file() and previous_backup is not None:
        shutil.copy2(previous_backup, backup)
        print("Imported optimizer backup:", previous_backup)
    if not best.is_file() and previous_best is not None:
        shutil.copy2(previous_best, best)
        print("Imported best checkpoint:", previous_best)

    payload = dict(config)
    if backup.is_file():
        payload["resume_checkpoint_path"] = str(backup)
        print("Will resume", config["experiment_name"], "from", backup)
    path.write_text(json.dumps(payload, indent=2))
    return path

MLP_CONFIG_PATH = write_config(mlp_config)
GAT_CONFIG_PATH = write_config(gat_config)
print(MLP_CONFIG_PATH)
print(GAT_CONFIG_PATH)


## Time-bounded training and best-checkpoint evaluation

In [ ]:
import shlex
import subprocess
import time

TRAIN_SCRIPT = REPO_DIR / "scripts" / "train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"

def base_training_command(config_path):
    if GPU_COUNT > 1:
        return [
            sys.executable,
            "-m",
            "torch.distributed.run",
            "--standalone",
            f"--nproc_per_node={GPU_COUNT}",
            str(TRAIN_SCRIPT),
            "--config",
            str(config_path),
        ]
    return [
        sys.executable,
        str(TRAIN_SCRIPT),
        "--config",
        str(config_path),
    ]

def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=run_environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()

def run_experiment(config, config_path, budget):
    # Refresh immediately before each attempt so rerunning this cell resumes a
    # backup created by a previous interrupted attempt.
    config_path = write_config(config)
    experiment = config["experiment_name"]
    run_dir = RESULTS_DIR / experiment
    summary_path = run_dir / "summary.json"
    if summary_path.is_file():
        existing = json.loads(summary_path.read_text())
        evaluation_checkpoint = existing.get("resolved_config", {}).get(
            "evaluation_checkpoint_path"
        )
        if evaluation_checkpoint is None:
            print(experiment, "already completed normally; skipping.")
            return
        print(experiment, "has a timeout evaluation; resuming training.")

    log_path = run_dir / "training.log"
    command = [
        "timeout",
        "--signal=INT",
        "--kill-after=5m",
        budget,
        *base_training_command(config_path),
    ]
    started = time.time()
    return_code = run_and_stream(command, log_path)
    elapsed = time.time() - started
    print(experiment, "training return code:", return_code)
    print(experiment, "training wall seconds:", round(elapsed, 1))

    if return_code == 0 and summary_path.is_file():
        print("Training completed normally; result bundle already exists.")
        return

    best_checkpoint = (
        Path(config["checkpoint_dir"])
        / f"{experiment}_checkpoint.pt"
    )
    assert best_checkpoint.is_file(), (
        f"No best checkpoint exists for {experiment}: {best_checkpoint}"
    )
    print("Evaluating best checkpoint:", best_checkpoint)
    evaluation_command = [
        sys.executable,
        str(TRAIN_SCRIPT),
        "--config",
        str(config_path),
        "--evaluate-checkpoint",
        str(best_checkpoint),
    ]
    evaluation_code = run_and_stream(
        evaluation_command,
        run_dir / "evaluation.log",
    )
    assert evaluation_code == 0, (
        f"Best-checkpoint evaluation failed with code {evaluation_code}"
    )

    # Keep resume provenance even if the checked-out train.py predates the
    # evaluation-history restoration helper.
    checkpoint = torch.load(
        best_checkpoint,
        map_location="cpu",
        weights_only=True,
    )
    training_state = checkpoint.get("training_state", {})
    if training_state and summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        summary["training_history"] = training_state.get("history", [])
        summary["best_epoch"] = training_state.get(
            "best_epoch",
            checkpoint.get("epoch"),
        )
        summary["best_metric"] = training_state.get("best_metric")
        summary_path.write_text(json.dumps(summary, indent=2))


In [ ]:
if RUN_MLP_CONTROL:
    run_experiment(mlp_config, MLP_CONFIG_PATH, MLP_TRAIN_BUDGET)
else:
    print("Skipping long MLP control by configuration.")


In [ ]:
run_experiment(gat_config, GAT_CONFIG_PATH, GAT_TRAIN_BUDGET)


## Compare and package results

In [ ]:
import pandas as pd

rows = []
for config in (mlp_config, gat_config):
    if config is mlp_config and not RUN_MLP_CONTROL:
        continue
    run_dir = RESULTS_DIR / config["experiment_name"]
    metrics_path = run_dir / "metrics.csv"
    summary_path = run_dir / "summary.json"
    if not metrics_path.is_file():
        print("Missing metrics:", metrics_path)
        continue

    metrics = pd.read_csv(metrics_path)
    if "kernel_family" in metrics:
        metrics = metrics[metrics["kernel_family"] == "all"]
    summary = json.loads(summary_path.read_text())
    for split in ("test", "exemplar"):
        selected = metrics[metrics["split"] == split]
        rows.append(
            {
                "experiment": config["experiment_name"],
                "split": split,
                "macro_smape": selected["smape"].mean(),
                "macro_r2": selected["r2"].mean(),
                "best_epoch": summary.get("best_epoch"),
                "best_validation_smape": summary.get("best_metric"),
            }
        )

comparison = pd.DataFrame(rows)
display(comparison)

archive = Path(
    __import__("shutil").make_archive(
        str(WORK / "ll_hls4ml_long_run_results"),
        "zip",
        root_dir=RESULTS_DIR,
    )
)
print("Packaged results:", archive)

from IPython.display import FileLink
display(FileLink(str(archive)))


## Interpretation

- Treat this as a **capacity and topology comparison**, not a hyperparameter
  search.
- The primary question is whether the three-layer GATv2 clearly exceeds the matched
  pooled MLP after both receive substantially longer optimization.
- Differences of only a few SMAPE points remain inconclusive at one seed.
- If GATv2 is promising, the next rented-GPU study should repeat the matched pair
  over at least three seeds before testing context, heads, or deeper hierarchy.
- If GATv2 does not beat the MLP, inspect learning curves and optimization before
  concluding topology is unhelpful. Normalized attention and shallow global
  pooling still impose known priors that may be poor for additive resources and
  critical paths.
